In [50]:
import os 
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

In [51]:
load_dotenv()

True

In [52]:
groq_api_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

In [53]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

In [54]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")

documents = loader.load()
print(documents)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content="HR POLICY DOCUMENT\n=================\n\nCompany: Nexora Technologies Pvt. Ltd.\nEffective Date: 08/10/2026\nVersion: 1.0\n\n1. PURPOSE\n-----------\nThis HR policy defines the workplace rules, employee responsibilities,\nbenefits, and procedures of Nexora Technologies Pvt. Ltd. The purpose is to\nmaintain a professional, respectful, inclusive, safe, and productive\nworkplace.\n\n2. WORKING HOURS\n----------------\n- Standard working hours are 9:00 AM to 6:00 PM, Monday to Friday.\n- Employees are expected to be punctual and available during working hours.\n- Employees are entitled to a one-hour lunch break.\n- Remote or flexible work arrangements require manager approval.\n\n3. ATTENDANCE AND LEAVE\n-----------------------\n- Employees must inform their manager as early as possible if they cannot\n  attend work.\n- Planned leave should be requested and approved in advance.\n- Emergency leave should be reported as soon

In [55]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(len(chunks))

19


In [56]:
embeddings_model = JinaEmbeddings(jina_api_key=jina_api_key, model_name="jina-embeddings-v2-base-en")

In [59]:
vectorstore = FAISS.from_documents(chunks, embeddings_model)

In [68]:
retriver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

def hr_policy_search(query):
    """Search for relevant HR policies based on a query."""
    top_matches = retriver.invoke(query)
    return top_matches

In [ ]:
# query = "What is the company's policy on remote work?"

# top_matches = vectorstore.similarity_search(query, k=3)

# for i, match in enumerate(top_matches):
#     print(f"Match {i+1}:")
#     print(match.page_content)
#     print()

In [70]:
from langchain_groq import ChatGroq

llm = ChatGroq(groq_api_key=groq_api_key,temperature=0, model_name="openai/gpt-oss-20b")


In [71]:
from langchain.agents import create_agent
hr_assistant = create_agent(model=llm, tools=[hr_policy_search], system_prompt="""
you are a helpful HR assistant. always use the hr_policy_search tool to answer questions about the company's HR policies. if you don't know the answer, say "I don't know" and do not make up an answer.
""")

In [73]:
response = hr_assistant.invoke({
    "messages": [
        {"role": "user", "content": "What is the company's policy on remote work?"}
    ]
})

In [85]:
print(response["messages"][-1].content)

**Remote Work Policy (Section 11)**  

- **Eligibility & Agreement** – Employees must have an approved remote‑work arrangement.  
- **Working Hours & Availability** – Remote workers must keep the agreed working hours and be reachable during those times.  
- **Data & Equipment** – Company data and equipment must be protected. Use only secure networks and approved applications.  
- **Meetings & Collaboration** – Remote employees are required to attend all scheduled meetings and maintain the same level of participation as office‑based staff.  
- **Professional Environment** – The home workspace should remain professional and free from distractions.  
- **Performance & Conduct** – Remote workers must adhere to the same performance standards and conduct expectations as on‑site employees.  

If you need more detail (e.g., eligibility criteria, equipment provision, or specific security protocols), let me know!
